# [16.2] KernelSHAP and PartitionSHAP Controls - Solutions

Runs the local reference implementation, all visible tests, and the committed CUDA verification-report assertions.

In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter16_shapley_attribution_baselines"
section = "part2_kernelshap_partition_shap_controls"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_kernelshap_partition_shap_controls.tests as tests
from chapter16_shapley_attribution_baselines.exercises.part2_kernelshap_partition_shap_controls import solutions


## Visible tests

In [ ]:
tests.test_kernelshap_kernel_weight_uses_finite_coalition_formula(
    solutions.kernelshap_kernel_weight,
)
tests.test_kernelshap_approximation_report_matches_exact_additive_game(
    solutions.kernelshap_approximation_report,
    solutions.additive_game,
)
tests.test_kernelshap_interaction_report_splits_conjunction_credit(
    solutions.kernelshap_approximation_report,
    solutions.conjunction_game,
)
tests.test_partition_shap_report_recovers_additive_groups(
    solutions.partition_shap_report,
    solutions.additive_game,
)
tests.test_partition_shap_report_splits_interaction_group_symmetrically(
    solutions.partition_shap_report,
    solutions.conjunction_game,
)
tests.test_partition_shap_report_rejects_invalid_grouping(
    solutions.partition_shap_report,
    solutions.additive_game,
)
tests.test_notebook_contract(solutions.run_smoke_test)
tests.test_committed_report_has_real_cuda_kernel_partition_evidence()


## Smoke-test contract

In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["kernel_additive"]["approximates_exact"], "KernelSHAP additive parity should pass."
assert contract["kernel_additive"]["shapley_values"] == [1.0, -2.0, 0.5], "KernelSHAP should recover additive weights."
assert contract["kernel_interaction"]["approximates_exact"], "KernelSHAP interaction parity should pass."
assert contract["partition_additive"]["recovers_exact"], "PartitionSHAP additive parity should pass."
assert contract["partition_additive"]["group_values"] == [3.0, 7.0], "Partition groups should receive additive sums."
assert contract["partition_interaction"]["player_values"] == [0.5, 0.5], "Interaction-group credit should split evenly."
contract


## Committed CUDA verification report

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
evidence = report["metrics"]["gpu_evidence"]

assert report["accepted"] is True, "The committed 16.2 report should be accepted."
assert report["tests_passed"] is True, "The report should have no known failures."
assert report["gt_tier"] == "GT-0", "16.2 should stay scoped to GT-0 finite-table controls."
assert evidence["uses_cuda"] is True, "The report should contain real CUDA evidence."
assert evidence["placeholder_only"] is False, "The report should not be placeholder-only."
assert gpu["cuda_available"] is True, "CUDA should be available for the report."
assert gpu["preflight_passed"] is True, "The CUDA SHAP preflight should pass."
assert gpu["kernel_approximates_exact"] is True, "KernelSHAP should match exact Shapley on model ablations."
assert gpu["kernel_max_abs_error"] <= 1e-8, "KernelSHAP full-table regression error should stay tiny."
assert gpu["kernel_vs_true_max_abs_error"] <= 1e-4, "KernelSHAP should match analytic ground truth."
assert gpu["partition_recovers_exact"] is True, "Singleton PartitionSHAP should recover exact values."
assert gpu["aligned_partition_recovers_exact"] is True, "Grouped PartitionSHAP should use exact Owen values."
assert gpu["mismatched_partition_recovers_exact"] is True, "Mismatched groups should still be exact Owen values."
assert gpu["cross_group_partition_recovers_exact"] is True, "The cross-group interaction control should recover exact values."
assert gpu["cross_group_irrelevant_credit"] <= 1e-8, "The irrelevant player should receive zero cross-group interaction credit."
assert gpu["shuffled_control_rejected"] is True, "The shuffled-label attribution control should be rejected."
assert gpu["within_vram_budget"] is True, "The report should stay within the VRAM budget."
{
    "device": gpu["device"],
    "kernel_vs_true_max_abs_error": gpu["kernel_vs_true_max_abs_error"],
    "partition_recovers_exact": gpu["partition_recovers_exact"],
    "cross_group_irrelevant_credit": gpu["cross_group_irrelevant_credit"],
    "shuffled_control_rejected": gpu["shuffled_control_rejected"],
    "peak_vram_gb": gpu["peak_vram_gb"],
}
